In [1]:
### IMPORTAÇÃO DE BIBLIOTECAS ####

import pandas as pd
import numpy
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pathlib import Path
import time
import datetime as dt
import warnings

user = "wconceicao"
password = "zJm7$j%qRU@WoCxM"
host = "dw-ro.data.grupoa.education"
port = "5432"
database = "postgres"

password_encoded = quote_plus(password)

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}"
)

start_date = '2025-09-01'
end_date = '2025-09-09'
campaigns = (1605, 1553, 1025, 1700)

In [2]:
def executar_qry(query,nome_df):

    inicio = time.time()

    try:
        print(f'Executando: {nome_df}...')
        df = pd.read_sql(text(query),engine)

        tempo_total = time.time() - inicio
        minutos = int(tempo_total // 60)
        segundos = tempo_total % 60

        if minutos > 0:
            tempo = f'{minutos}min {segundos:.1f}s'
        else:
            tempo = f'{segundos:.2f}s' 

        num_linhas = len(df)

        print(f'✅ {nome_df}: {num_linhas} linhas | ⏱️{tempo}')

        return df       

    except Exception as e:
        tempo_total = time.time() - inicio
        print(f' Erro em {nome_df} após {tempo_total:.2f}s: {e}')
        return None

In [3]:
qry_sispag = """
    WITH Sispag AS (
    SELECT
        CASE
            WHEN vd_produto.tipoproduto = 'C' THEN 'ARTMED360'
            ELSE 'SECAD'
        END AS "nome_ies",

        -- Limpeza de celular simplificada com Regex
        RIGHT(REGEXP_REPLACE(vd_cliente.celular, '\D', '', 'g'), 11) AS "celular",

        vd_produto.tipoproduto::text,

        -- Ajuste de Timezone
        CASE
            WHEN vd_compra."codVendedor" IS NULL OR vd_compra."codVendedor"::text = '8000' 
                THEN DATE(vd_compra.datahora AT TIME ZONE 'UTC' AT TIME ZONE 'America/Sao_Paulo')
            ELSE DATE(vd_compra.datahora)
        END AS dt,

        TO_CHAR(DATE_TRUNC('hour', vd_compra.datahora), 'HH24:MI') AS hr,
        vd_compra.compraid AS id,
        vd_compraitem.valor * vd_compra.valortotal / NULLIF(SUM(vd_compraitem.valor) OVER (PARTITION BY vd_compra.compraid), 0) AS value,

        CASE
            WHEN vd_compra."codVendedor" IS NULL THEN 'eCommerce'
            WHEN LEFT(vd_compra."codVendedor"::text, 1) = '8' THEN 'Representantes'
            WHEN vd_compra."codVendedor"::text IN ('9186', '9323', '9326', '9325') THEN 'Receptivo'
            WHEN LEFT(vd_compra."codVendedor"::text, 1) = 'R' THEN 'Renovacao'
            ELSE 'Call Center'
        END AS channel,

        CASE
        WHEN programs.area IS NOT NULL THEN programs.area
        ELSE
            CASE
                WHEN vd_produto.nomeresumido::text = ANY (ARRAY['PROURGEM'::character varying::text, 'PROENDÓCRINO'::character varying::text, 'PRO-UROLOGIA'::character varying::text, 'PROTERAPÊUTICA'::character varying::text]) THEN 'Medicina'::text
                WHEN vd_produto.nomeresumido::text = 'PROMEVET'::text THEN 'Veterinária'::text
                WHEN vd_produto.nomeresumido::text = 'PROENF-URG'::text THEN 'Enfermagem'::text
                WHEN vd_produto.nomeresumido::text = ANY (ARRAY['PROFISIO/NEURO'::character varying::text, 'PROFISIO/TRAUMA'::character varying::text]) THEN 'Fisioterapia'::text
                ELSE NULL::text
            END
    END AS area
        
    FROM app_sispag_pagamento.vd_compra
    LEFT JOIN app_sispag_pagamento.vd_compraitem ON vd_compra.compraid = vd_compraitem.compraid
    LEFT JOIN app_sispag_pagamento.vd_produto ON vd_compraitem.produtoid = vd_produto.produtoid
    LEFT JOIN app_sispag_pagamento.vd_request ON vd_compra.requestid = vd_request.requestid
    LEFT JOIN app_sispag_pagamento.vd_cliente ON vd_compra.clienteid = vd_cliente.clienteid
    LEFT JOIN bu_secad.programs ON vd_produto.nomeresumido::text = programs.program
    

    WHERE vd_compra.datahora >= '2026-02-01 00:00:00' 
      AND vd_compra.datahora <= '2026-03-01 23:59:59'
      AND vd_produto.tipoproduto::text IN ('P', 'C')
      AND vd_request.ambiente::text = 'P'
      AND (vd_compra."codVendedor"::text <> '123' OR vd_compra."codVendedor" IS NULL)
      AND LOWER(vd_cliente.nome::text) NOT LIKE '%teste%'
)


SELECT *
FROM Sispag  
WHERE channel NOT IN ('eCommerce','Representantes')
  AND tipoproduto = 'P'
ORDER BY dt DESC;

"""

qry_olos = """

WITH tab_olos AS (
    SELECT
        right(regexp_replace(phone_number, '\D', '', 'g'), 11) AS phone_olos,
        customer_id,
        disposition_id,
        disposition_nivel_1,

        start_agent_date::date AS data_olos,
        start_agent_date::time(0) AS hora_olos,

        ROW_NUMBER() OVER (
            PARTITION BY right(regexp_replace(phone_number, '\D', '', 'g'), 11)
            ORDER BY start_agent_date DESC
        ) AS rn,

        CASE 
            WHEN disposition_nivel_1 ~* 'agendado|outra plataforma' THEN 0
            WHEN disposition_nivel_1 ~* 'nao localizado|bolsa gratuita' THEN 15
            WHEN disposition_nivel_1 ~* 'matricula|financeiro|outra formacao|curso de interesse|outra ies|analise de disciplinas|metodologia' THEN 30
            WHEN disposition_nivel_1 ~* 'ligacao falhando|ligacao caiu|sem audio|nao informou|nao atende|apresentacao|metodo pagamento|intresse em evento' THEN 10
            WHEN disposition_nivel_1 ~* 'inadimplente' THEN 60
            WHEN disposition_nivel_1 ~* 'semestre|nao quer EAD|sem tempo' THEN 180
            ELSE 7
        END AS retorno_em_dias,

        CASE 
            WHEN disposition_nivel_1 ~* 'fora do publico|nao acionar|desconhece|numero errado|sem afinidade|sem interesse'
                THEN 'Não retornar'
            ELSE 'retornar'
        END AS status_retorno,

        EXTRACT(EPOCH FROM wrap_duration) AS tempo_segundos,
        tablename

    FROM integration_operations.vw_call_center_calls
    WHERE campaign_id IN (1025,1553,1605,1690)
      AND start_agent_date::date >= '2026-02-01'
      AND start_agent_date::date <=  '2026-03-01'  
      AND wrap_duration IS NOT NULL
      AND EXTRACT(EPOCH FROM wrap_duration) > 0
)
SELECT
    phone_olos,
    customer_id,
    disposition_nivel_1,
    data_olos,
    hora_olos,
    tablename,
    tempo_segundos
FROM tab_olos
WHERE rn = 1


"""


qry_blip = """

WITH blip_hist AS (
    SELECT 
        template,

        CASE
            WHEN template ~* 'ativo|ativa' THEN 'Ativo'
            WHEN template ~* 'inativo|inativa|troca' THEN 'Inativo'
            WHEN template ~* 'cancelado' THEN 'Cancelado'
            WHEN template ~* 'page' THEN 'Page View'
            WHEN template ~* 'evento' THEN 'Evento'
            WHEN template ~* 'carrinho' THEN 'Carrinho'
            ELSE 'Outros'
        END AS base_type,

        CASE
            WHEN template ~* 'Medicina|proemped|proneuroped|propsicomed|proterapeutica' THEN 'Medicina'
            WHEN template ~* 'Fisioterapia|profisio' THEN 'Fisioterapia'
            WHEN template ~* 'Psicologia|psicologo|proneuropsi' THEN 'Psicologia'
            WHEN template ~* 'Enfermagem' THEN 'Enfermagem'
            WHEN template ~* 'Pediatria' THEN 'Pediatria'
            WHEN template ~* 'page|evento' THEN 'Base Leads'
            ELSE 'Não Identificado'
        END AS area,
        
        CASE 
            WHEN tags ~* 'Parou de responder|REL_|SEM_TAG|IMPRODUTIVO|TRANSF_|automatica' THEN 'IMPRODUTIVO'
            WHEN tags ~* 'retorno|negociacao|fechado final' THEN 'PRODUTIVO'
            WHEN tags ~* 'Sem interesse|Recusa' THEN 'RECUSA'
            WHEN tags ~* 'Matricula|boleto' THEN 'VENDA'
            ELSE ''
        END AS status,
        
        RIGHT(regexp_replace(contact_id, '\D', '', 'g'), 11) AS contact_id,
        first_client_message,

        REPLACE(REPLACE(REPLACE(tags, '["', ''), '"]', ''), '"', '') AS tags,

        attendance_started_at AS data_inicio_ts,
        close_date AS data_fim_ts,

        attendance_started_at::date AS data_inicio,
        close_date::date AS data_fim,

        ROW_NUMBER() OVER(
            PARTITION BY RIGHT(regexp_replace(contact_id, '\D', '', 'g'), 11)
            ORDER BY close_date DESC NULLS LAST
        ) AS rn

    FROM integration_operations.blip_details_conversations
    WHERE template ~* 'cap_ops'
      AND account_id = 'secadativo'
      AND attendance_started_at::date >= '2026-02-01'
      AND attendance_started_at::date <= '2026-03-01'
),

blip_leadtime AS (
    SELECT 
        template,
        base_type,
        area,
        status,
        contact_id,
        first_client_message,
        tags,
        data_inicio,
        data_fim,
        (data_fim - data_inicio) AS leadtime_dias_inteiro,
        EXTRACT(EPOCH FROM (data_fim_ts - data_inicio_ts)) AS leadtime_seg,
        rn
    FROM blip_hist
)

SELECT *
FROM blip_leadtime
WHERE rn = 1;





"""

<>:10: SyntaxWarning: invalid escape sequence '\D'
<>:74: SyntaxWarning: invalid escape sequence '\D'
<>:162: SyntaxWarning: invalid escape sequence '\D'
<>:10: SyntaxWarning: invalid escape sequence '\D'
<>:74: SyntaxWarning: invalid escape sequence '\D'
<>:162: SyntaxWarning: invalid escape sequence '\D'
C:\Users\wconceicao\AppData\Local\Temp\ipykernel_18900\444447132.py:10: SyntaxWarning: invalid escape sequence '\D'
  RIGHT(REGEXP_REPLACE(vd_cliente.celular, '\D', '', 'g'), 11) AS "celular",
C:\Users\wconceicao\AppData\Local\Temp\ipykernel_18900\444447132.py:74: SyntaxWarning: invalid escape sequence '\D'
  right(regexp_replace(phone_number, '\D', '', 'g'), 11) AS phone_olos,
C:\Users\wconceicao\AppData\Local\Temp\ipykernel_18900\444447132.py:162: SyntaxWarning: invalid escape sequence '\D'
  RIGHT(regexp_replace(contact_id, '\D', '', 'g'), 11) AS contact_id,


In [4]:
queries ={
    'sispag':qry_sispag,
    'tab_olos':qry_olos,
    'tab_blip':qry_blip,
}

dfs = {}

try:
    for nome, qry in queries.items():
        dfs[nome] = executar_qry(qry,nome)
except Exception as e:
    print(f'Erro ao executar as queries: {e}')
finally:
    breakpoint()    


Executando: sispag...
✅ sispag: 367 linhas | ⏱️3.64s
Executando: tab_olos...
✅ tab_olos: 6244 linhas | ⏱️1min 13.8s
Executando: tab_blip...
✅ tab_blip: 5235 linhas | ⏱️24.48s


In [5]:
df_sispag  = dfs['sispag']
df_olos = dfs['tab_olos']
df_blip = dfs['tab_blip']

In [6]:
sispag = df_sispag.copy()
olos = df_olos.copy()
blip = df_blip.copy()

In [7]:
sispag_merge = sispag.merge(
    olos[['phone_olos','disposition_nivel_1','tablename','tempo_segundos','data_olos','hora_olos']],
    left_on='celular',
    right_on='phone_olos',
    how='left'
)
sispag_merge = sispag_merge.merge(
    blip[['contact_id','tags','template','data_inicio','data_fim']],
    left_on='celular',
    right_on='contact_id',
    how='left'
)
sispag_merge = sispag_merge.fillna('-')

In [8]:
sispag_merge['qtde_olos'] = ((sispag_merge['phone_olos'] != '-') & (sispag_merge['tablename'] != '-')).astype(int)
sispag_merge['qtde_blip'] = ((sispag_merge['contact_id'] != '-') & (sispag_merge['template'] != '-')).astype(int)


In [12]:
vendas_agg = sispag_merge.groupby('area').agg(
    total = ('celular','count'),
    total_olos = ('qtde_olos','sum'),
    total_blip = ('qtde_blip','sum')
).reset_index()
vendas_agg['nao_loc'] = vendas_agg['total'] - (vendas_agg['total_olos'] + vendas_agg['total_blip'])
vendas_agg['tx_olos'] = (vendas_agg['total_olos'] / vendas_agg['total'] * 100).round(1)
vendas_agg['tx_blip'] = (vendas_agg['total_blip'] / vendas_agg['total'] * 100).round(1)
vendas_agg['tx_nao_loc'] = (vendas_agg['nao_loc'] / vendas_agg['total'] * 100).round(1)
vendas_agg = vendas_agg[[
    'area',
    'total',
    'total_olos','tx_olos',
    'total_blip','tx_blip',
    'nao_loc','tx_nao_loc',
]]
vendas_agg

,area,total,total_olos,tx_olos,total_blip,tx_blip,nao_loc,tx_nao_loc
0,-,69,21,30.4,32,46.4,16,23.2
1,Enfermagem,44,10,22.7,4,9.1,30,68.2
2,Fisioterapia,110,33,30.0,15,13.6,62,56.4
3,Medicina,79,27,34.2,8,10.1,44,55.7
4,Multidisciplinar,18,5,27.8,1,5.6,12,66.7
5,Nutrição,4,2,50.0,1,25.0,1,25.0
6,Saúde Mental,34,14,41.2,6,17.6,14,41.2
7,Veterinária,9,7,77.8,0,0.0,2,22.2
